# Séance 4 — Hyperparamètres, régularisation et pipelines

Après la comparaison des modèles et l'introduction de la validation croisée, nous allons apprendre à **choisir les hyperparamètres**.

Objectifs :
- comprendre paramètres / hyperparamètres ;
- étudier Ridge, Lasso et ElasticNet ;
- utiliser `GridSearchCV` et `RandomizedSearchCV` ;
- comprendre la normalisation ;
- construire un `Pipeline` ;
- éviter la fuite d'information lors de la validation croisée.

Fil rouge : `load_diabetes()`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import loguniform

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 1. Paramètres et hyperparamètres

Un **paramètre** est appris à partir des données : par exemple les coefficients β d'une régression.

Un **hyperparamètre** est choisi avant l'apprentissage : par exemple `alpha` dans Ridge/Lasso ou `n_neighbors` dans KNN.

La validation croisée permet de comparer différentes valeurs d'hyperparamètres sans utiliser le jeu de test.

## 2. Régularisation

Ridge ajoute une pénalisation L2 :

$
\text{MSE}+\alpha\sum_j\beta_j^2
$

Lasso ajoute une pénalisation L1 :

$
\text{MSE}+\alpha\sum_j|\beta_j|
$

ElasticNet combine L1 et L2.

$
\text{MSE}+\alpha\left(\rho\sum_j|\beta_j|+(1-\rho)\sum_j\beta_j^2\right).
$

L'idée générale est de limiter la complexité du modèle afin d'améliorer sa généralisation.

In [ ]:
alphas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]

res = []
for alpha in alphas:
    scores = cross_val_score(
        Ridge(alpha=alpha), Xtr, ytr, cv=5, scoring="r2"
    )
    res.append([alpha, scores.mean(), scores.std()])

ridge_df = pd.DataFrame(res, columns=["alpha","R2_moyen","R2_ecart_type"])
ridge_df

In [ ]:
plt.figure(figsize=(8,5))
plt.semilogx(ridge_df["alpha"], ridge_df["R2_moyen"], marker="o")
plt.xlabel("alpha")
plt.ylabel("R² moyen")
plt.title("Ridge : choix de alpha")
plt.show()

### Question

Que se passe-t-il lorsque `alpha` devient très grand ? Pourquoi une pénalisation trop forte peut-elle être problématique ?

In [ ]:
for alpha in [1e-5, 0.01, 0.1, 1, 10]:
    modele = Lasso(alpha=alpha, max_iter=10000)
    modele.fit(Xtr, ytr)
    print("\nalpha =", alpha)
    print(pd.Series(modele.coef_, index=diabetes.feature_names))

### Lasso et sélection de variables

Lasso peut mettre certains coefficients exactement à zéro. Il peut donc réaliser une forme de sélection de variables.

Mais « coefficient nul » ne signifie pas automatiquement « variable inutile » : le résultat dépend de la pénalisation et des autres variables.

In [ ]:
gcv = GridSearchCV(
    Lasso(max_iter=10000),
    param_grid={"alpha": [10**p for p in range(-5, 3)]},
    cv=5,
    scoring="r2"
)

gcv.fit(Xtr, ytr)

print("Meilleur alpha :", gcv.best_params_)
print("Meilleur R² CV :", gcv.best_score_)

In [ ]:
pd.DataFrame(gcv.cv_results_)[
    ["param_alpha","mean_test_score","std_test_score","rank_test_score"]
].sort_values("rank_test_score")

## 3. `RandomizedSearchCV`

`GridSearchCV` teste toutes les combinaisons proposées.

`RandomizedSearchCV` tire un nombre donné de configurations dans des distributions. Il devient intéressant lorsque l'espace de recherche est grand.

In [ ]:
random_search = RandomizedSearchCV(
    Ridge(),
    param_distributions={"alpha": loguniform(1e-5, 100)},
    n_iter=20,
    cv=5,
    scoring="r2",
    random_state=42
)

random_search.fit(Xtr, ytr)

random_search.best_params_, random_search.best_score_

## 4. Normalisation

De nombreux algorithmes sont sensibles à l'échelle des variables, notamment KNN.

La standardisation est :

$
Z=\frac{X-\mu}{\sigma}
$

On **apprend** μ et σ sur le train, puis on applique ces valeurs au test.

In [ ]:
scaler = StandardScaler()

Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

print("Moyennes train :", Xtr_scaled.mean(axis=0))
print("Ecarts-types train :", Xtr_scaled.std(axis=0))

`fit_transform` = apprendre la transformation puis l'appliquer.

`transform` = appliquer une transformation déjà apprise.

Il ne faut pas faire `fit_transform(Xte)` : cela utiliserait des informations du test.

## 5. Pipeline

Un Pipeline permet de faire :

**normalisation → modèle**

dans une seule procédure.

C'est essentiel avec la validation croisée : à chaque fold, le scaler est ajusté uniquement sur les observations d'apprentissage du fold.

In [ ]:
assemblage = Pipeline(
    steps=[
        ("normalisation", StandardScaler()),
        ("prediction", Ridge())
    ]
)

scores = cross_val_score(
    assemblage, Xtr, ytr, cv=5, scoring="r2"
)

print(scores)
print("Moyenne :", scores.mean())
print("Écart-type :", scores.std())

In [ ]:
gcv_ridge = GridSearchCV(
    assemblage,
    param_grid={
        "prediction__alpha": [1e-5, 1e-3, 1e-2, 0.1, 1, 10, 100]
    },
    cv=5,
    scoring="r2"
)

gcv_ridge.fit(Xtr, ytr)

gcv_ridge.best_params_, gcv_ridge.best_score_

### Pourquoi `prediction__alpha` ?

Dans un Pipeline, la syntaxe :

`nom_de_l_etape__parametre`

permet d'accéder au paramètre d'une étape.

Ici, `prediction` est le nom de l'étape et `alpha` est le paramètre de Ridge.

In [ ]:
knn_pipeline = Pipeline(
    steps=[
        ("normalisation", StandardScaler()),
        ("prediction", KNeighborsRegressor())
    ]
)

gcv_knn = GridSearchCV(
    knn_pipeline,
    param_grid={
        "prediction__n_neighbors": [1,3,5,10,20,40,80]
    },
    cv=5,
    scoring="r2"
)

gcv_knn.fit(Xtr, ytr)

gcv_knn.best_params_, gcv_knn.best_score_

### Question finale

Pourquoi le Pipeline est-il préférable à une normalisation effectuée une fois sur tout `X` avant la validation croisée ?

**Bilan :** nous savons maintenant rechercher des hyperparamètres et intégrer proprement les étapes de préparation dans la validation croisée.